### Incluir los nombres de los colectores en la tabla Persons

Dado que los nombres de colectores fueron insertados en la tabla Occurrences como texto, es necesario generar las entradas homologadas en la tabla Persons y asociar dichas entradas a la tabla Occurrences usando el PersonID. En este sentido, este Notebook es de único uso.

Las homologaciones de las grafías redundantes de nombres de colectores fueron realizadas en LibreOffice Calc y se encuentran en el archivo `collectors.csv`.

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from datetime import datetime

import credentials

In [2]:
conn_str = 'mysql+mysqlconnector://' + \
	f"{credentials.mysql['username']}:{credentials.mysql['password']}" + \
	'@localhost:3306/Mutis'
engine = create_engine(conn_str)
connection = engine.connect()

In [3]:
colls = pd.read_csv("collectors.csv", delimiter="\t")
colls["Standarized_Collector"] = colls.Standarized_Collector.str.replace(".", ". "
	).str.replace(",", ", "
	).str.replace(r"\.\s+,", ".,", regex=True
	).str.replace(r"\s+", " ", regex=True
	).str.replace(r"\s+$|^\s+", "", regex=True
	)

In [51]:
persons = pd.read_sql_table("Persons", connection)

# 1

Si no se han ingresado ningún colector de la tabla a la base de datos, ejecute las siguiente celdas, de lo contrario salte a 2

In [54]:
uniqcolls = colls[
	~colls.Standarized_Collector.isin(persons.NameVerbatim)
	].groupby("Standarized_Collector"
	).size(
	).reset_index(
	).drop(columns=0)

startidx = persons.PersonID.max() + 1
uniqcolls["PersonID"] = uniqcolls.index + startidx
uniqcolls[["LastName", "FirstName"]] = uniqcolls.Standarized_Collector.str.split(
	r",\s+", regex=True, expand=True
	)
uniqcolls[["Abbreviation", "Affiliation", "Email"]] = None, None, None
uniqcolls = uniqcolls.rename(columns={"Standarized_Collector":"NameVerbatim"})
uniqcolls["TimeStamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# Check no duplicates with records already in db

for row in uniqcolls.itertuples():
	q = persons.loc[persons.LastName == row.LastName]
	if q.shape[0] > 0:
		print(row.NameVerbatim)
		print(persons.loc[persons.LastName == row.LastName, "NameVerbatim"].values)
		print("=" * 40)

Barbosa, O.
['Barbosa, C.']
Betancur, A.
['Betancur, Julio']
Castellanos, J. D.
['Castellanos, C.']
Cuellar, Y.
['Cuellar, D. M.']
Daniel, R.
['Daniel, H.']
Escobar, W.
['Escobar, L. K.']
Gamboa, C.
['Gamboa, M. A.']
Gamboa, M. C.
['Gamboa, M. A.']
García, A.
['García, Y.']
García, B.
['García, Y.']
García, C.
['García, Y.']
García, F.
['García, Y.']
García, G.
['García, Y.']
García, H.
['García, Y.']
García, L. A. de
['García, Y.']
García, M.
['García, Y.']
García, M. C.
['García, Y.']
García, N.
['García, Y.']
García, Ni.
['García, Y.']
García, R.
['García, Y.']
García, Roger Fabián
['García, Y.']
García, Yamile
['García, Y.']
Gil, C. E.
['Gil, E.']
Hernández, G.
['Hernández, A.']
Hernández, J.
['Hernández, A.']
Hernández, L.
['Hernández, A.']
Hernández, M.
['Hernández, A.']
Hernández, N.
['Hernández, A.']
Hernández, Nicolás
['Hernández, A.']
Lewis, W. H.
['Lewis, G. P.']
Mayorga, C.
['Mayorga, M.' 'Mayorga, N. C.']
Montenegro, A.
['Montenegro, P. A.']
Montenegro, E.
['Montenegro, P.

In [57]:
uniqcolls[persons.columns].to_sql('Persons', engine, if_exists='append', index=False, 
		chunksize=10000, method='multi'
	)

1193

# 2

Se obtienen los PersonID de la BD Mutis que corresponden a todos los colectores verbatim de la tabla ocurrencias.

In [4]:
persons = pd.read_sql_table("Persons", connection)

In [ ]:
# Just to check there aren't missing PersonIDs in the original set of collector names

print(
colls.merge(
	persons[["NameVerbatim", "PersonID"]],
	left_on="Standarized_Collector",
	right_on="NameVerbatim",
	how="left"
).to_string()
)

                              CollectorVerbatim                      Standarized_Collector                               NameVerbatim  PersonID
0                                     Acero, E.                             Acero, Enrique                             Acero, Enrique       171
1                                   Acero, J.D.                               Acero, J. D.                               Acero, J. D.       172
2                            Acevedo-Luna, N.I.                        Acevedo-Luna, N. I.                        Acevedo-Luna, N. I.       175
3                                Acevedo, E. de                             Acevedo, E. de                             Acevedo, E. de       173
4                                   Acevedo, O.                                Acevedo, O.                                Acevedo, O.       174
5                          Acosta-Arteaga, C.E.                      Acosta-Arteaga, C. E.                      Acosta-Arteaga, C. E.   

In [229]:
peoper = pd.read_sql_table("PeoplePersons", connection)

In [230]:
people = pd.read_sql_table("People", connection)

In [225]:
maxper = startidx - 1
maxpeople = people.PeopleID.max()

In [226]:
maxper, maxpeople

(170, 153)

In [227]:
# The following works because all collectors are one-person teams
peopleid = maxpeople

for perid in persons.PersonID[persons.PersonID > maxper]:
	peopleid += 1

	s = text("INSERT INTO People(PeopleID) VALUES (:ppid)")
	connection.execute(s, {"ppid": peopleid})
	connection.commit()

	s = text("INSERT INTO PeoplePersons VALUES (:ppid, :perid, 1, '2026-02-16')")
	connection.execute(s, {"ppid": peopleid, "perid": perid})
	connection.commit()


In [234]:
occcolls = pd.read_sql_query("SELECT CollectorVerbatim, Collector FROM Occurrences WHERE CollectorVerbatim IS NOT NULL", connection)

In [238]:
colls

,CollectorVerbatim,NameVerbatim,LastName,FirstName,PersonID,Abbreviation,Affiliation,Email,TimeStamp
0,"Acero, E.","Acero, Enrique",Acero,Enrique,171,None,None,None,2026-02-16
1,"Acero, J.D.","Acero, J. D.",Acero,J. D.,172,None,None,None,2026-02-16
2,"Acevedo-Luna, N.I.","Acevedo-Luna, N. I.",Acevedo-Luna,N. I.,173,None,None,None,2026-02-16
3,"Acevedo, E. de","Acevedo, E. de",Acevedo,E. de,174,None,None,None,2026-02-16
4,"Acevedo, O.","Acevedo, O.",Acevedo,O.,175,None,None,None,2026-02-16
...,...,...,...,...,...,...,...,...,...
1325,"Zucolotto, S.","Zucolotto, S.",Zucolotto,S.,1496,None,None,None,2026-02-16
1326,"Zuluaga-C., Juliana","Zuluaga-C., Juliana",Zuluaga-C.,Juliana,1497,None,None,None,2026-02-16
1327,"Zuluaga-R., S.","Zuluaga-R., S.",Zuluaga-R.,S.,1498,None,None,None,2026-02-16
1328,"Zulueta, I. de","Zulueta, I. de",Zulueta,I. de,1499,None,None,None,2026-02-16


In [237]:
persons[persons.NameVerbatim.str.startswith('Acero')]

,PersonID,NameVerbatim,FirstName,LastName,Abbreviation,Affiliation,Email,TimeStamp
170,171,"Acero, Enrique",Enrique,Acero,None,None,None,2026-02-16
171,172,"Acero, J. D.",J. D.,Acero,None,None,None,2026-02-16


In [232]:
people

,PeopleID,TimeStamp
0,1,2025-06-18 09:15:52
1,2,2025-06-18 09:15:52
2,3,2025-06-18 09:16:06
3,4,2025-06-18 09:16:06
4,5,2025-06-18 09:16:06
...,...,...
1478,1479,2026-02-16 20:17:50
1479,1480,2026-02-16 20:17:50
1480,1481,2026-02-16 20:17:50
1481,1482,2026-02-16 20:17:50


In [246]:
colls[colls.NameVerbatim == "Lancheros, Héctor Orlando"]

,CollectorVerbatim,NameVerbatim,LastName,FirstName,PersonID,Abbreviation,Affiliation,Email,TimeStamp
677,"Lancheros, H.","Lancheros, Héctor Orlando",Lancheros,Héctor Orlando,848,None,None,None,2026-02-16
678,"Lancheros, Héctor O.","Lancheros, Héctor Orlando",Lancheros,Héctor Orlando,849,None,None,None,2026-02-16
679,"Lancheros, Héctor Orlando","Lancheros, Héctor Orlando",Lancheros,Héctor Orlando,850,None,None,None,2026-02-16


In [244]:
for row in occcolls.itertuples():

	if pd.isna(row.Collector):

		try:

			stdcoll = colls.loc[
				colls.CollectorVerbatim == row.CollectorVerbatim,
				"NameVerbatim"].item()


			perid = persons.loc[
				persons.NameVerbatim == stdcoll, 
				"PersonID"
				].item()
			
		except:
			#print(f"{perid=}")
			print(f"{row.CollectorVerbatim=}")
			print(f"{stdcoll=}")
			continue

		peopleid = peoper.loc[
			(peoper.Person == perid),# & (peoper.Order == 1), 
			"People"].item()

		#print(f"{perid=}, {peopleid=}")

		s = text("SELECT OccurrenceID FROM Occurrences where CollectorVerbatim = :thcoll")
		thoccs = connection.execute(s, {"thcoll": row.CollectorVerbatim}).fetchall()

		#print(f"{thoccs}")

		#for i in (x[0] for x in thoccs):

		#	s = text("UPDATE Occurrences SET Collector = :ppid WHERE OccurrenceID = :occid")
		#	connection.execute(s, {"ppid": peopleid, "occid": i})
		#	connection.commit()

		#break


row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumnos Instituto de La Salle'
stdcoll='Alumnos La Salle'
row.CollectorVerbatim='Alumn

KeyboardInterrupt: 

In [149]:
colls.loc[colls.NameVerbatim == collector, "CollectorVerbatim"].item()

'Acero, E.'

In [223]:
connection.close()